# Honest validation of a 60 m sprint kinematics pipeline

**What survives leave-one-out cross-validation, and where in the stride it lives.**

Published sprint-kinematics models report high in-sample fit: R2 = 0.795
(Vellucci & Beaudette 2023, PCA into stepwise regression, n = 40) and R2 = 0.92
(Griffiths, MIT Sloan 2026, video-driven simulation into gradient boosting).
Neither holds the feature-selection step out of the validation loop. This
notebook measures, on one 30-athlete cohort, what each common shortcut is worth
and what remains once every choice is moved inside the fold.

The notebook is organised as independent levels, each a self-contained analysis
that consumes the same cohort object:

| Level | Question | Tag |
|---|---|---|
| 1 | Is the headline variable measuring what it claims? | `V*` |
| 2 | What does each analytic shortcut inflate the score by? | `L*` |
| 3 | Where in the stride cycle do the surviving effects sit? | `S*` |
| 4 | What is the honest predictive result? | `W*` |

Conventions carried from the pipeline: blue = faster; knee flexion positive
(ISB, Wu 2002); SI units; every number recomputed from the C3D data, never
copied. `04_Audit_Provenance.ipynb` remains the descriptive source of truth;
this notebook adds the validation accounting and does not modify the pipeline.

## 1. Setup

Imports, a single constants block, and a results directory. Every threshold that
governs a decision downstream is named here rather than buried in a cell.

In [1]:
from __future__ import annotations

import itertools
import warnings
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.interpolate import CubicSpline
from sklearn.linear_model import RidgeCV

import sprint_pipeline as SP

# --- Analysis constants ----------------------------------------------------
N_PERMUTATIONS: int = 5_000            # shuffles for every permutation test
COHORT_STRIDE_COUNT: int = 5           # strides averaged per athlete at top speed
CYCLE_POINTS: int = 101                # samples on the normalised 0-100% cycle
N_ANGLES: int = 13                     # sagittal joint angles per frame

# Miyashiro & Nagahara 2019 (n=79), converted to the ISB flexion-positive scale
PUBLISHED_PEAK_KNEE_FLEXION_DEG: float = 148.4

# Fixed regression guard: this trial must not move under a refactor
REFERENCE_TRIAL_FRAMES: int = 531
REFERENCE_TRIAL_PEAK_MS: float = 8.506

# SPM is run on a PRE-SPECIFIED angle set; testing all 13 and reporting the best
# would reintroduce the very selection leak Level 2 measures
PRESPECIFIED_SPM_ANGLES: Tuple[str, ...] = ("knee_R", "hip_R", "trunk_lean")
SPM_ALPHA: float = 0.05

SPLINE_HALF_WINDOW: int = 4            # frames either side of a peak for [V1]
SMOOTHING_PENALTY: float = 1.0         # B-spline roughness penalty for angle curves
GROUND_CONTACT_BAND_M: float = 0.03    # foot within 3 cm of local ground = stance
STRIDE_COUNT_SWEEP: Tuple[int, ...] = (5, 8, 10, 999)  # 999 = every valid stride

RESULTS_DIR: Path = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# A shared ridge factory, so every ridge model in the notebook is identical
def make_ridge() -> RidgeCV:
    """Return a fresh RidgeCV with the pipeline's standard alpha grid."""
    return RidgeCV(alphas=np.logspace(-3, 4, 60))

warnings.filterwarnings("ignore", category=RuntimeWarning)
print("Setup complete "
      f"| {N_PERMUTATIONS:,} permutations | cohort stride count = {COHORT_STRIDE_COUNT}")

Setup complete | 5,000 permutations | cohort stride count = 5


## 2. Data loading

One pass over the C3D files. Each trial becomes one `AthleteRecord`; the loader
is defensive because a single unreadable trial should not abort the cohort.
Everything downstream reads from the returned list, so the expensive marker
processing happens exactly once.

In [2]:
@dataclass
class AthleteRecord:
    """All per-athlete quantities the validation levels need.

    Arrays follow the pipeline's conventions: `raw_angles` is (frames, 13) in
    degrees, and every *_cycle is (101, 13) on the phase-normalised axis with
    0% at foot contact.
    """
    pid: str
    peak_velocity_ms: float                     # smoothed peak = regression target
    raw_peak_velocity_ms: float                 # unsmoothed peak, kept for [L5]
    body_height_m: float
    n_frames: int
    contact_time_s: float
    stride_time_s: float
    raw_angles: np.ndarray                       # (frames, 13)
    knee_flexion_3d: np.ndarray                  # (frames,) full 3-D knee angle
    smoothed_velocity_curve: np.ndarray          # (frames,)
    stride_windows: List[Tuple[int, int]]        # valid windows, nearest-peak first
    angle_cycle: np.ndarray                      # (101, 13) unsmoothed top-speed mean
    stance_mask: np.ndarray                      # (101,) bool, True = ground contact
    top_phase_spread: float                      # peak-flexion phase scatter, %
    accel_alt_cycle: Optional[np.ndarray] = None  # alternating steps 1-2-3
    accel_par_cycle: Optional[np.ndarray] = None  # same-parity steps 1-3-5
    accel_alt_spread: Optional[float] = None
    accel_par_spread: Optional[float] = None
    smoothed_cycle: np.ndarray = field(default=None, repr=False)  # filled post-load


def _full_3d_knee_flexion(raw64: np.ndarray) -> np.ndarray:
    """Knee flexion as the unsigned 3-D angle between thigh and shank vectors.

    Args:
        raw64: (frames, 64, 3) cleaned marker array for one trial.

    Returns:
        (frames,) knee flexion in degrees, 0 = straight leg. This bypasses the
        sagittal projection used by `compute_joint_angles`, so comparing the two
        isolates how much the projection costs (see [V2]).
    """
    thigh = raw64[:, SP.MK_R_KNEE].mean(axis=1) - raw64[:, SP.MK_R_HIP]
    shank = raw64[:, SP.MK_R_ANKLE].mean(axis=1) - raw64[:, SP.MK_R_KNEE].mean(axis=1)
    cosine = (thigh * shank).sum(axis=1) / (
        np.linalg.norm(thigh, axis=1) * np.linalg.norm(shank, axis=1))
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))


def _stance_mask_for(raw64: np.ndarray, windows: Sequence[Tuple[int, int]]) -> np.ndarray:
    """Fraction-of-cycle stance mask, averaged over the given stride windows.

    A frame is stance when the lower of heel/toe height sits within
    `GROUND_CONTACT_BAND_M` of that trial's local ground (2nd percentile).
    """
    vertical = np.minimum(raw64[:, SP.MK_R_HEEL, SP._VERT],
                          raw64[:, SP.MK_R_TOE, SP._VERT])
    ground = np.percentile(vertical, 2)
    per_window = [SP.resample_to_cycle(
        (vertical[start:end + 1] < ground + GROUND_CONTACT_BAND_M).astype(float))
        for start, end in windows]
    return np.mean(per_window, axis=0) > 0.5


def _windows_nearest_peak(marker_data: np.ndarray, peak_frame: int
                          ) -> List[Tuple[int, int]]:
    """Every valid top-speed stride window, ordered nearest-to-peak first."""
    windows, _rejected = SP.find_top_speed_strides(marker_data, peak_frame,
                                                    n_strides=999)
    if not windows:
        return []
    midpoints = np.array([(start + end) / 2 for start, end in windows])
    order = np.argsort(np.abs(midpoints - peak_frame))
    return [windows[i] for i in order]


def load_cohort(verbose: bool = True) -> List[AthleteRecord]:
    """Read every eligible C3D trial into an `AthleteRecord`.

    Args:
        verbose: print per-stage progress and a final shape summary.

    Returns:
        List of records, one per athlete that produced a valid top-speed cycle.
        Trials that fail to load or segment are logged and skipped, never fatal.
    """
    records: List[AthleteRecord] = []
    skipped: List[Tuple[str, str]] = []

    for filepath in sorted(SP.C3D_DIR.glob("*.c3d")):
        pid = filepath.stem.split("-")[0].strip()
        if pid in SP.EXCLUDED_PIDS:
            continue
        try:
            trial = SP.clean_for_angles(filepath)
            sample_rate = trial["fs"]
            raw64 = trial["raw64"]
            peak_frame = trial["peak_frame"]

            angle_cycle, _n_used, top_spread = SP.stride_cycle_angles(
                raw64, trial["mk"], peak_frame)
            if angle_cycle is None:
                skipped.append((pid, "no valid top-speed stride"))
                continue
            if angle_cycle.shape != (CYCLE_POINTS, N_ANGLES):
                skipped.append((pid, f"unexpected cycle shape {angle_cycle.shape}"))
                continue

            raw_angles, _names = SP.compute_joint_angles(raw64)
            raw_velocity = SP.compute_velocity(trial["mk"], sample_rate, raw64=raw64)[0]
            peak_velocity, smoothed_velocity = SP.peak_velocity_smoothed(raw_velocity)

            cohort_windows, _ = SP.find_top_speed_strides(trial["mk"], peak_frame)
            stance_mask = _stance_mask_for(raw64, cohort_windows)
            contact_time = float(np.mean([
                (np.minimum(raw64[start:end + 1, SP.MK_R_HEEL, SP._VERT],
                            raw64[start:end + 1, SP.MK_R_TOE, SP._VERT])
                 < np.percentile(np.minimum(raw64[:, SP.MK_R_HEEL, SP._VERT],
                                            raw64[:, SP.MK_R_TOE, SP._VERT]), 2)
                 + GROUND_CONTACT_BAND_M).sum() / sample_rate
                for start, end in cohort_windows]))
            stride_time = float(np.mean([(end - start) / sample_rate
                                         for start, end in cohort_windows]))

            record = AthleteRecord(
                pid=pid,
                peak_velocity_ms=peak_velocity,
                raw_peak_velocity_ms=trial["peak_vel_ms"],
                body_height_m=SP.PARTICIPANT_ANTHRO[pid]["body_height"],
                n_frames=trial["n_frames"],
                contact_time_s=contact_time,
                stride_time_s=stride_time,
                raw_angles=raw_angles,
                knee_flexion_3d=_full_3d_knee_flexion(raw64),
                smoothed_velocity_curve=smoothed_velocity,
                stride_windows=_windows_nearest_peak(trial["mk"], peak_frame),
                angle_cycle=angle_cycle,
                stance_mask=stance_mask,
                top_phase_spread=top_spread,
            )

            # Acceleration, two codings of the same block-exit steps
            alt_steps, _ = SP.find_first_steps(trial["mk"], n_steps=3)
            if len(alt_steps) == 3:
                cycle, _n, spread = SP.stride_cycle_angles(
                    raw64, trial["mk"], peak_frame, windows=alt_steps)
                record.accel_alt_cycle, record.accel_alt_spread = cycle, spread
            parity_steps, _ = SP.find_first_steps(trial["mk"], n_steps=6)
            if len(parity_steps) >= 5:
                cycle, _n, spread = SP.stride_cycle_angles(
                    raw64, trial["mk"], peak_frame,
                    windows=[parity_steps[0], parity_steps[2], parity_steps[4]])
                record.accel_par_cycle, record.accel_par_spread = cycle, spread

            records.append(record)
            if verbose:
                print(f"  {pid:6} loaded | {trial['n_frames']:4d} frames "
                      f"| peak {peak_velocity:5.2f} m/s "
                      f"| {len(record.stride_windows):2d} valid strides")
        except Exception as error:                       # noqa: BLE001 - log, continue
            skipped.append((pid, str(error)))
            if verbose:
                print(f"  {pid:6} SKIPPED: {error}")

    if verbose:
        print(f"\nLoaded {len(records)} athletes | skipped {len(skipped)}")
        for pid, reason in skipped:
            print(f"  skipped {pid}: {reason}")
    return records

In [3]:
cohort = load_cohort(verbose=True)

# Cohort-wide B-spline smoothing is per-athlete-per-angle and independent across
# athletes, so it is applied once to the stacked array and written back.
_stacked = np.stack([record.angle_cycle for record in cohort])
_smoothed = SP.smooth_angle_curves(_stacked, penalty=SMOOTHING_PENALTY)
for record, smoothed in zip(cohort, _smoothed):
    record.smoothed_cycle = smoothed

print(f"\nStacked cohort cycles: {_stacked.shape} (athletes, points, angles)")

  SB061  loaded |  474 frames | peak  8.86 m/s | 14 valid strides


  SB101  loaded |  454 frames | peak  9.30 m/s | 16 valid strides


  SB102  loaded |  546 frames | peak  7.61 m/s | 19 valid strides


  SB110  loaded |  579 frames | peak  7.31 m/s | 19 valid strides


  SB111  loaded |  505 frames | peak  8.32 m/s | 17 valid strides


...

  SB80   loaded |  575 frames | peak  7.14 m/s | 18 valid strides


  SB81   loaded |  562 frames | peak  7.38 m/s | 18 valid strides


  SB82   loaded |  468 frames | peak  9.13 m/s | 17 valid strides


  SB91   loaded |  542 frames | peak  7.64 m/s | 15 valid strides


  SB92   loaded |  545 frames | peak  7.60 m/s | 15 valid strides

Loaded 30 athletes | skipped 0

Stacked cohort cycles: (30, 101, 13) (athletes, points, angles)


### Cohort accessors

Small helpers turn the record list into the arrays and per-pid maps each level
expects. Keeping them in one place means no level re-derives a target vector.

In [4]:
def pids_of(records: Sequence[AthleteRecord]) -> List[str]:
    """Athlete IDs in stable sorted order."""
    return [record.pid for record in records]


def velocity_vector(records: Sequence[AthleteRecord]) -> np.ndarray:
    """(n,) smoothed peak velocity, the regression target."""
    return np.array([record.peak_velocity_ms for record in records])


def height_vector(records: Sequence[AthleteRecord]) -> np.ndarray:
    """(n,) measured stature in metres."""
    return np.array([record.body_height_m for record in records])


def smoothed_cycle_map(records: Sequence[AthleteRecord]) -> Dict[str, np.ndarray]:
    """pid -> (101, 13) smoothed top-speed cycle."""
    return {record.pid: record.smoothed_cycle for record in records}


def raw_cycle_map(records: Sequence[AthleteRecord]) -> Dict[str, np.ndarray]:
    """pid -> (101, 13) unsmoothed top-speed cycle."""
    return {record.pid: record.angle_cycle for record in records}


def hilo_cycle_map(records: Sequence[AthleteRecord]) -> Dict[str, np.ndarray]:
    """pid -> (101, 11) HI/LO-recoded cycle (limbs ordered by range of motion)."""
    return {record.pid: SP.recode_limbs(record.angle_cycle) for record in records}


def scalar_feature_frame(records: Sequence[AthleteRecord]) -> pd.DataFrame:
    """One row per athlete of the 22 scalar features, exactly as the ML stage."""
    return pd.DataFrame([SP.scalar_features(record.angle_cycle) for record in records],
                        index=pids_of(records))


def target_by_pid(records: Sequence[AthleteRecord]) -> Dict[str, float]:
    """pid -> smoothed peak velocity, for the SPM label-shuffling test."""
    return {record.pid: record.peak_velocity_ms for record in records}


PIDS = pids_of(cohort)
VELOCITY = velocity_vector(cohort)
HEIGHT = height_vector(cohort)
FEATURES = scalar_feature_frame(cohort)
print(f"feature frame: {FEATURES.shape} | target range "
      f"{VELOCITY.min():.2f}-{VELOCITY.max():.2f} m/s (sd {VELOCITY.std():.2f})")

feature frame: (30, 22) | target range 6.39-9.35 m/s (sd 0.71)


### Data validation

Three cheap checks before any analysis: cohort size, no missing values, and the
fixed reference trial. If any fails, later numbers are not trustworthy.

In [5]:
def validate_cohort(records: Sequence[AthleteRecord], features: pd.DataFrame,
                    velocity: np.ndarray) -> None:
    """Assert cohort size, absence of NaNs, and the reference-trial invariant.

    Raises:
        AssertionError: on the wrong athlete count or any NaN feature/target.
    """
    assert len(records) == 30, f"expected 30 athletes, got {len(records)}"
    n_missing = int(features.isna().sum().sum())
    assert n_missing == 0, f"{n_missing} missing feature values"
    assert not np.isnan(velocity).any(), "missing target values"

    reference = [record for record in records
                 if record.n_frames == REFERENCE_TRIAL_FRAMES]
    print(f"[check] cohort size          : {len(records)} athletes  OK")
    print(f"[check] missing values       : {n_missing}  OK")
    print(f"[check] target sd            : {velocity.std():.3f} m/s "
          f"(expected ~0.72)")
    for record in reference:
        match = abs(record.raw_peak_velocity_ms - REFERENCE_TRIAL_PEAK_MS) < 5e-4
        print(f"[check] reference {record.pid:5}      : {record.n_frames} frames, "
              f"{record.raw_peak_velocity_ms:.3f} m/s  "
              f"{'OK' if match else 'REGRESSION'}")
    assert reference, "reference 531-frame trial not found"


validate_cohort(cohort, FEATURES, VELOCITY)

[check] cohort size          : 30 athletes  OK
[check] missing values       : 0  OK
[check] target sd            : 0.711 m/s (expected ~0.72)
[check] reference SB25       : 531 frames, 8.506 m/s  OK


## Level 1 - Measurement validity

The headline predictor is knee range of motion, and measured peak knee flexion
here (~132 deg) sits ~16 deg below published video work (~148 deg). Before
trusting any correlation, that gap has to be explained. If it were caused by the
60 Hz sampling missing a brief flexion peak, the miss would be **worse for faster
athletes**, whose limbs travel further between frames - which would by itself
manufacture the finding that faster athletes flex less.

### [V1] Does 60 Hz sampling miss the flexion peak, and does the miss scale with speed?

In [6]:
def assess_knee_sampling_loss(records: Sequence[AthleteRecord]) -> pd.DataFrame:
    """Peak knee flexion lost to 60 Hz sampling, per athlete.

    A cubic spline through the frames around each stride's peak gives the
    continuous maximum; the gap to the sampled maximum is what the frame rate
    misses. The gap is then correlated with velocity and stride frequency.

    Returns:
        DataFrame with sampled/spline peaks, the miss, and per-athlete velocity
        and stride frequency.
    """
    knee_index = SP.ANGLE_NAMES.index("knee_R")
    rows = []
    for record in records:
        knee = record.raw_angles[:, knee_index]
        sampled_peaks, spline_peaks = [], []
        for start, end in record.stride_windows[:COHORT_STRIDE_COUNT]:
            local_peak = start + int(np.argmax(knee[start:end + 1]))
            if local_peak - SPLINE_HALF_WINDOW < 0 or \
               local_peak + SPLINE_HALF_WINDOW >= len(knee):
                continue
            support = np.arange(local_peak - SPLINE_HALF_WINDOW,
                                local_peak + SPLINE_HALF_WINDOW + 1)
            dense = CubicSpline(support, knee[support])(
                np.linspace(support[0], support[-1], 801))
            sampled_peaks.append(knee[local_peak])
            spline_peaks.append(float(dense.max()))
        if not sampled_peaks:
            continue
        rows.append({
            "pid": record.pid,
            "sampled_peak_deg": float(np.mean(sampled_peaks)),
            "spline_peak_deg": float(np.mean(spline_peaks)),
            "missed_deg": float(np.mean(spline_peaks) - np.mean(sampled_peaks)),
            "velocity_ms": record.peak_velocity_ms,
            "stride_frequency_hz": 1.0 / record.stride_time_s,
        })
    return pd.DataFrame(rows)


sampling_loss = assess_knee_sampling_loss(cohort)
mean_sampled = sampling_loss.sampled_peak_deg.mean()
r_velocity, p_velocity = stats.pearsonr(sampling_loss.missed_deg,
                                        sampling_loss.velocity_ms)
r_frequency, p_frequency = stats.pearsonr(sampling_loss.missed_deg,
                                          sampling_loss.stride_frequency_hz)

print(f"[V1] sampled peak flexion   = {mean_sampled:6.2f} deg (SD "
      f"{sampling_loss.sampled_peak_deg.std():.2f})")
print(f"[V1] spline-recovered peak  = {sampling_loss.spline_peak_deg.mean():6.2f} deg")
print(f"[V1] missed by 60 Hz        = {sampling_loss.missed_deg.mean():6.3f} deg "
      f"(max {sampling_loss.missed_deg.max():.3f})")
print(f"[V1] shortfall vs published = {PUBLISHED_PEAK_KNEE_FLEXION_DEG - mean_sampled:6.1f} deg")
print(f"[V1] r(miss, velocity)      = {r_velocity:+.3f}  p={p_velocity:.4f}")
print(f"[V1] r(miss, stride freq)   = {r_frequency:+.3f}  p={p_frequency:.4f}")
print("[V1] --> the miss is ~0.2 deg and unrelated to speed, so the 16 deg gap")
print("[V1]     is a definition offset, NOT a speed-dependent sampling artefact.")
sampling_loss.to_csv(RESULTS_DIR / "v1_knee_sampling_loss.csv", index=False)

[V1] sampled peak flexion   = 132.36 deg (SD 8.52)
[V1] spline-recovered peak  = 132.57 deg
[V1] missed by 60 Hz        =  0.213 deg (max 0.503)
[V1] shortfall vs published =   16.0 deg
[V1] r(miss, velocity)      = -0.000  p=0.9997
[V1] r(miss, stride freq)   = +0.180  p=0.3424
[V1] --> the miss is ~0.2 deg and unrelated to speed, so the 16 deg gap
[V1]     is a definition offset, NOT a speed-dependent sampling artefact.


### [V2] Is the gap the 2-D sagittal projection, and does the choice change the finding?

In [7]:
def compare_projection_definitions(records: Sequence[AthleteRecord]) -> pd.DataFrame:
    """Knee peak and ROM under the 2-D sagittal vs full 3-D definition.

    Returns:
        DataFrame with both peak-flexion and both range-of-motion measures per
        athlete, so the effect on the velocity correlation can be read directly.
    """
    knee_index = SP.ANGLE_NAMES.index("knee_R")
    rows = []
    for record in records:
        sagittal = record.raw_angles[:, knee_index]
        three_d = record.knee_flexion_3d
        windows = record.stride_windows[:COHORT_STRIDE_COUNT]
        rows.append({
            "pid": record.pid,
            "peak_2d_deg": float(np.mean([sagittal[a:b + 1].max() for a, b in windows])),
            "peak_3d_deg": float(np.mean([three_d[a:b + 1].max() for a, b in windows])),
            "rom_2d_deg": float(np.mean(
                [np.ptp(SP.resample_to_cycle(sagittal[a:b + 1])) for a, b in windows])),
            "rom_3d_deg": float(np.mean(
                [np.ptp(SP.resample_to_cycle(three_d[a:b + 1])) for a, b in windows])),
        })
    return pd.DataFrame(rows)


projection = compare_projection_definitions(cohort)
r_2d, p_2d = stats.pearsonr(projection.rom_2d_deg, VELOCITY)
r_3d, p_3d = stats.pearsonr(projection.rom_3d_deg, VELOCITY)

print(f"[V2] peak flexion, 2-D sagittal = {projection.peak_2d_deg.mean():6.1f} deg")
print(f"[V2] peak flexion, full 3-D     = {projection.peak_3d_deg.mean():6.1f} deg")
print(f"[V2] projection accounts for      "
      f"{projection.peak_3d_deg.mean() - projection.peak_2d_deg.mean():+6.1f} deg "
      f"(wrong sign -> projection is not the cause)")
print(f"[V2] knee ROM 2-D vs velocity   = {r_2d:+.3f}  p={p_2d:.4f}")
print(f"[V2] knee ROM 3-D vs velocity   = {r_3d:+.3f}  p={p_3d:.4f}")
print(f"[V2] r(ROM_2d, ROM_3d)          = "
      f"{np.corrcoef(projection.rom_2d_deg, projection.rom_3d_deg)[0, 1]:+.3f} "
      f"-> the finding is robust to the definition")

[V2] peak flexion, 2-D sagittal =  132.4 deg
[V2] peak flexion, full 3-D     =  130.3 deg
[V2] projection accounts for        -2.0 deg (wrong sign -> projection is not the cause)
[V2] knee ROM 2-D vs velocity   = -0.565  p=0.0012
[V2] knee ROM 3-D vs velocity   = -0.617  p=0.0003
[V2] r(ROM_2d, ROM_3d)          = +0.957 -> the finding is robust to the definition


### [V3] The reference-trial invariant

In [8]:
def verify_reference_trial(records: Sequence[AthleteRecord]) -> bool:
    """Confirm the fixed 531-frame / 8.506 m/s trial is unchanged.

    Returns:
        True if exactly the expected trial matches to 3 decimal places.
    """
    matches = [record for record in records
               if record.n_frames == REFERENCE_TRIAL_FRAMES
               and abs(record.raw_peak_velocity_ms - REFERENCE_TRIAL_PEAK_MS) < 5e-4]
    for record in matches:
        print(f"[V3] {record.pid}: {record.n_frames} frames, "
              f"{record.raw_peak_velocity_ms:.3f} m/s")
    all_frames = np.array([record.n_frames for record in records])
    print(f"[V3] cohort frames: mean {all_frames.mean():.1f} "
          f"(range {all_frames.min()}-{all_frames.max()})")
    print(f"[V3] reference invariant: {'PASS' if matches else 'FAIL'}")
    return bool(matches)


verify_reference_trial(cohort)

[V3] SB25: 531 frames, 8.506 m/s
[V3] cohort frames: mean 520.1 (range 452-655)
[V3] reference invariant: PASS


True

## Level 2 - The leakage ledger

Each entry compares the **reported** number a common practice produces against
the **honest** number from the same data with every choice moved inside the
cross-validation fold. Rows accumulate in `LEDGER_ROWS` and are tabulated in
Level 4.

In [9]:
LEDGER_ROWS: List[Dict[str, float]] = []


def record_ledger_entry(label: str, reported: float, honest: float) -> None:
    """Append one shortcut's reported/honest pair to the running ledger."""
    LEDGER_ROWS.append({"shortcut": label, "reported": reported,
                        "honest": honest, "inflation": reported - honest})


def _top_k_by_abs_correlation(features: np.ndarray, target: np.ndarray,
                              k: int) -> np.ndarray:
    """Indices of the k features most correlated (absolute) with the target."""
    correlations = np.array([abs(np.corrcoef(features[:, col], target)[0, 1])
                             for col in range(features.shape[1])])
    return np.argsort(-correlations)[:k]

### [L1] Selecting the top-k features on all 30, cross-validating only the regression

The textbook leak: rank features by their correlation with speed on the whole
cohort, then honestly leave-one-out the regression. The ranking has already seen
every athlete.

In [10]:
def measure_selection_leakage(features: pd.DataFrame, target: np.ndarray,
                              k_values: Sequence[int]) -> pd.DataFrame:
    """Reported vs honest LOO R2 for top-k univariate feature selection.

    Args:
        features: (n, p) scalar feature frame.
        target: (n,) velocity.
        k_values: numbers of features to select.

    Returns:
        DataFrame with the all-data and inside-fold scores and their gap.
    """
    matrix = features.values
    rows = []
    for k in k_values:
        selected = _top_k_by_abs_correlation(matrix, target, k)
        reported = SP.loo_r2_fast(matrix[:, selected], target)

        def select_inside_fold(train_idx: np.ndarray, test_idx: np.ndarray,
                               k: int = k) -> Tuple[np.ndarray, np.ndarray]:
            chosen = _top_k_by_abs_correlation(matrix[train_idx], target[train_idx], k)
            return matrix[np.ix_(train_idx, chosen)], matrix[np.ix_(test_idx, chosen)]

        honest = SP.fit_metrics(target, SP.loo_predict(
            None, target, None, select_inside_fold))["LOO_R2"]
        rows.append({"k": k, "reported": reported, "honest": honest,
                     "inflation": reported - honest})
    return pd.DataFrame(rows)


selection_leakage = measure_selection_leakage(FEATURES, VELOCITY, (1, 2, 3, 5))
print("[L1] top-k univariate selection:")
print(selection_leakage.to_string(index=False,
      formatters={c: "{:+.3f}".format for c in
                  ["reported", "honest", "inflation"]}))
_row = selection_leakage.set_index("k").loc[3]
record_ledger_entry("top-3 univariate selection on all 30",
                    _row.reported, _row.honest)

[L1] top-k univariate selection:

 k reported honest inflation
 1   +0.278 +0.278    -0.000
 2   +0.220 +0.220    -0.000
 3   +0.228 +0.072    +0.156
 5   +0.188 -0.046    +0.234


### [L2] Searching the model space, then reporting the winner's cross-validated score

The sharper, more realistic version. Every candidate is scored by leave-one-out,
so the analyst can honestly say each number was cross-validated - but the
**choice among 1,540 models** saw all 30 athletes. `loo_r2_fast` (closed-form LOO
for least squares, matching `loo_predict` to six decimals) makes the inside-fold
re-search affordable.

In [11]:
def measure_model_search_leakage(features: pd.DataFrame, target: np.ndarray,
                                 height: np.ndarray, n_features: int = 3
                                 ) -> Dict[str, object]:
    """Reported vs honest LOO R2 when the best k-feature model is searched.

    The reported score searches all combinations on the full cohort; the honest
    score repeats the entire search inside every fold. Height is always included.

    Returns:
        Dict with both scores, the inflation, the winning feature set, and how
        often the inner loop agreed on one model.
    """
    matrix = features.values
    columns = list(features.columns)
    combinations = list(itertools.combinations(range(len(columns)), n_features))

    def design_for(subset: Tuple[int, ...], rows: np.ndarray) -> np.ndarray:
        return np.column_stack([matrix[np.ix_(rows, list(subset))], height[rows]])

    all_rows = np.arange(len(target))
    scores = [SP.loo_r2_fast(design_for(subset, all_rows), target)
              for subset in combinations]
    best_index = int(np.argmax(scores))
    reported = scores[best_index]

    predictions = np.zeros(len(target))
    chosen_per_fold = []
    for test_idx in range(len(target)):
        train_idx = np.delete(all_rows, test_idx)
        inner = [SP.loo_r2_fast(design_for(subset, train_idx), target[train_idx])
                 for subset in combinations]
        winner = combinations[int(np.argmax(inner))]
        chosen_per_fold.append(winner)
        design = np.column_stack([np.ones(len(train_idx)),
                                  matrix[np.ix_(train_idx, list(winner))],
                                  height[train_idx]])
        coefficients = np.linalg.pinv(design) @ target[train_idx]
        predictions[test_idx] = np.r_[
            1.0, matrix[test_idx, list(winner)], height[test_idx]] @ coefficients
    honest = float(1 - ((target - predictions) ** 2).sum()
                   / ((target - target.mean()) ** 2).sum())

    top_choice, top_count = Counter(chosen_per_fold).most_common(1)[0]
    return {
        "n_models": len(combinations),
        "reported": reported,
        "honest": honest,
        "winner": [columns[i] for i in combinations[best_index]] + ["height"],
        "modal_choice": [columns[i] for i in top_choice] + ["height"],
        "modal_count": top_count,
    }


model_search = measure_model_search_leakage(FEATURES, VELOCITY, HEIGHT)
print(f"[L2] searched {model_search['n_models']} three-feature (+height) models")
print(f"[L2] best score, searched on all 30 = {model_search['reported']:+.3f}")
print(f"[L2]   winner: {model_search['winner']}")
print(f"[L2] same search inside every fold  = {model_search['honest']:+.3f}")
print(f"[L2] inflation                      = "
      f"{model_search['reported'] - model_search['honest']:+.3f}")
print(f"[L2] inner loop chose {model_search['modal_choice']}")
print(f"[L2]   in {model_search['modal_count']}/30 folds -> a STABLE choice fitted")
print("[L2]   to noise is still noise; stability does not make it honest.")
record_ledger_entry(f"exhaustive search, {model_search['n_models']} models",
                    model_search["reported"], model_search["honest"])

[L2] searched 1540 three-feature (+height) models
[L2] best score, searched on all 30 = +0.403
[L2]   winner: ['knee_lo_ROM', 'knee_lo_mean', 'elbow_hi_mean', 'height']
[L2] same search inside every fold  = -0.020
[L2] inflation                      = +0.423
[L2] inner loop chose ['knee_lo_ROM', 'knee_lo_mean', 'elbow_hi_mean', 'height']
[L2]   in 26/30 folds -> a STABLE choice fitted
[L2]   to noise is still noise; stability does not make it honest.


### [L3] Fitting the fPCA basis on all 30, then cross-validating the regression

In [12]:
def measure_projection_leakage(records: Sequence[AthleteRecord], target: np.ndarray,
                               n_components: int = 6) -> pd.DataFrame:
    """Reported vs honest LOO R2 for fPCA-score regression, per curve coding.

    The reported score fits the fPCA basis on all 30 athletes; the honest score
    refits it inside every fold. Both use ridge regression.
    """
    pids = pids_of(records)
    codings = {"R/L": raw_cycle_map(records), "HI/LO": hilo_cycle_map(records)}
    rows = []
    for name, curves in codings.items():
        reported = SP.fit_metrics(target, SP.loo_predict(
            SP.fpca_scores_leaky(curves, pids, n_components), target,
            make_ridge))["LOO_R2"]
        honest = SP.fit_metrics(target, SP.loo_predict(
            None, target, make_ridge,
            SP.fpca_fold_transform(curves, pids, n_components)))["LOO_R2"]
        rows.append({"coding": name, "reported": reported, "honest": honest,
                     "inflation": reported - honest})
    return pd.DataFrame(rows)


projection_leakage = measure_projection_leakage(cohort, VELOCITY)
print("[L3] fPCA-score regression, basis on all 30 vs refit in fold:")
print(projection_leakage.to_string(index=False,
      formatters={c: "{:+.3f}".format for c in
                  ["reported", "honest", "inflation"]}))
print("[L3] both codings lose to the cohort mean; the leak buys almost nothing")
print("[L3] because there is no fPCA signal to inflate.")
_worst = projection_leakage.iloc[projection_leakage.inflation.idxmax()]
record_ledger_entry(f"fPCA basis on all 30 ({_worst.coding})",
                    _worst.reported, _worst.honest)

[L3] fPCA-score regression, basis on all 30 vs refit in fold:
coding reported honest inflation
   R/L   -0.223 -0.188    -0.036
 HI/LO   -0.200 -0.139    -0.060
[L3] both codings lose to the cohort mean; the leak buys almost nothing
[L3] because there is no fPCA signal to inflate.


### [L4] Reading a component's variance share as predictive power

In [13]:
def describe_pc1(records: Sequence[AthleteRecord]) -> Dict[str, object]:
    """PC1 variance share, dominant angle, speed correlation, and the reason.

    Returns:
        Dict with PC1's explained variance, its largest loading share, its
        correlation with velocity, and the between/within SD ratios that decide
        which angle the standardised fPCA can see.
    """
    fpca = SP.run_angle_fpca(smoothed_cycle_map(records), n_components=6)
    loadings = fpca["loadings"][0]
    squared = (loadings ** 2).sum(axis=0)
    dominant = int(np.argmax(squared))
    velocity = velocity_vector(records)

    ratios = {}
    for name in ("trunk_lean", "hip_R", "knee_R"):
        column = np.array([record.angle_cycle[:, SP.ANGLE_NAMES.index(name)]
                           for record in records])
        within = column.std(axis=1).mean()
        between = column.mean(axis=1).std()
        ratios[name] = (within, between, between / within)
    return {
        "variance_share": fpca["explained_var"][0],
        "dominant_angle": SP.ANGLE_NAMES[dominant],
        "dominant_share": squared[dominant] / squared.sum() * 100,
        "velocity_correlation": float(np.corrcoef(fpca["scores"][:, 0], velocity)[0, 1]),
        "ratios": ratios,
    }


pc1 = describe_pc1(cohort)
print(f"[L4] PC1 explains {pc1['variance_share']:.1f}% of the variance")
print(f"[L4] and is {pc1['dominant_share']:.0f}% {pc1['dominant_angle']} by loading")
print(f"[L4] r(PC1 score, velocity) = {pc1['velocity_correlation']:+.3f}")
print(f"\n[L4] {'angle':12}{'within-cycle SD':>17}{'between-athlete SD':>20}{'ratio':>8}")
for name, (within, between, ratio) in pc1["ratios"].items():
    print(f"[L4] {name:12}{within:17.1f}{between:20.1f}{ratio:8.2f}")
print("[L4] PC1 goes to the angle with the largest between/within ratio (trunk")
print("[L4] lean); the knee, at ratio 0.12, is invisible to a standardised fPCA.")

[L4] PC1 explains 42.0% of the variance
[L4] and is 77% trunk_lean by loading
[L4] r(PC1 score, velocity) = -0.177

[L4] angle         within-cycle SD  between-athlete SD   ratio
[L4] trunk_lean                2.4                 4.1    1.72
[L4] hip_R                    23.2                 7.5    0.32
[L4] knee_R                   39.3                 4.6    0.12
[L4] PC1 goes to the angle with the largest between/within ratio (trunk
[L4] lean); the knee, at ratio 0.12, is invisible to a standardised fPCA.


### [L5] Taking the maximum of a numerically differentiated velocity signal

In [14]:
def measure_target_noise(records: Sequence[AthleteRecord]) -> Dict[str, float]:
    """How much the raw velocity peak overshoots the smoothed peak.

    Returns:
        Dict with the mean/spread of the inflation and its correlation with the
        true peak (a nonzero correlation means the bias is noise, not an offset).
    """
    inflation = np.array([record.raw_peak_velocity_ms - record.peak_velocity_ms
                          for record in records])
    true_peak = velocity_vector(records)
    correlation, _p = stats.pearsonr(inflation, true_peak)
    return {"mean": float(inflation.mean()), "min": float(inflation.min()),
            "max": float(inflation.max()), "sd": float(inflation.std()),
            "correlation_with_peak": float(correlation)}


target_noise = measure_target_noise(cohort)
print(f"[L5] raw peak runs {target_noise['mean']:+.2f} m/s hot "
      f"(range {target_noise['min']:+.2f} to {target_noise['max']:+.2f}, "
      f"SD {target_noise['sd']:.2f})")
print(f"[L5] r(inflation, true peak) = {target_noise['correlation_with_peak']:+.3f} "
      f"-> uneven across athletes, so it is noise, not a constant offset")

[L5] raw peak runs +0.52 m/s hot (range +0.26 to +0.83, SD 0.15)
[L5] r(inflation, true peak) = +0.083 -> uneven across athletes, so it is noise, not a constant offset


### [L6] A feature whose window length is set by the outcome

In [15]:
def _partial_correlation(first: np.ndarray, second: np.ndarray,
                         control: np.ndarray) -> Tuple[float, float]:
    """Correlation of `first` and `second` with `control` linearly removed."""
    first_residual = first - np.polyval(np.polyfit(control, first, 1), control)
    second_residual = second - np.polyval(np.polyfit(control, second, 1), control)
    return stats.pearsonr(first_residual, second_residual)


def measure_window_length_artefact(records: Sequence[AthleteRecord],
                                   angles: Sequence[str]) -> pd.DataFrame:
    """Stance-phase ROM vs velocity, before and after controlling window length.

    Stance ROM is measured over only the stance points of the cycle, and the
    number of those points is set by contact time, which is set by speed. The
    partial correlation removes that shared driver.
    """
    stance_points = np.array([record.stance_mask.sum() for record in records])
    velocity = velocity_vector(records)
    rows = []
    for angle in angles:
        index = SP.ANGLE_NAMES.index(angle)
        stance_rom = np.array([
            np.ptp(record.angle_cycle[record.stance_mask, index])
            for record in records])
        raw_r, raw_p = stats.pearsonr(stance_rom, velocity)
        partial_r, partial_p = _partial_correlation(
            stance_rom, velocity, stance_points.astype(float))
        rows.append({"angle": angle, "r_raw": raw_r, "p_raw": raw_p,
                     "r_partial": partial_r, "p_partial": partial_p,
                     "survives": raw_p < 0.05 and partial_p < 0.05})
    return pd.DataFrame(rows)


window_angles = ["hip_R", "hip_L", "knee_R", "knee_L", "ankle_R",
                 "shoulder_R", "shoulder_L"]
window_artefact = measure_window_length_artefact(cohort, window_angles)
_stance_points = np.array([record.stance_mask.sum() for record in cohort])
print(f"[L6] r(stance points, velocity) = "
      f"{stats.pearsonr(_stance_points, VELOCITY)[0]:+.3f} (window length tracks speed)")
print(window_artefact.to_string(index=False,
      formatters={c: "{:+.3f}".format for c in ["r_raw", "r_partial"]}))
_n_significant_before = int((window_artefact.p_raw < 0.05).sum())
_n_survive = int(window_artefact.survives.sum())
print(f"[L6] {_n_significant_before}/7 significant raw; {_n_survive}/7 survive the "
      f"window-length control")

[L6] r(stance points, velocity) = -0.541 (window length tracks speed)
     angle  r_raw    p_raw r_partial  p_partial  survives
     hip_R -0.419 0.021267    -0.002   0.992573     False
     hip_L -0.481 0.007190    -0.205   0.276608     False
    knee_R -0.357 0.052817    -0.055   0.773300     False
    knee_L -0.505 0.004425    -0.275   0.141467     False
   ankle_R -0.543 0.001930    -0.301   0.106363     False
shoulder_R -0.399 0.028778    -0.073   0.703401     False
shoulder_L -0.378 0.039351    -0.009   0.962426     False
[L6] 6/7 significant raw; 0/7 survive the window-length control


### [L7] Adding strides without checking they are the same movement

In [16]:
def icc_one_way(table: pd.DataFrame, column: str, group: str = "pid") -> float:
    """Shrout-Fleiss ICC(1,1): between-athlete share of a feature's variance.

    ICC never sees the outcome, so screening features on it cannot leak.
    """
    grouped = table.groupby(group)[column]
    mean_replicates = grouped.size().mean()
    n_groups = grouped.ngroups
    between = mean_replicates * ((grouped.mean() - table[column].mean()) ** 2).sum() \
        / (n_groups - 1)
    within = sum(((values - values.mean()) ** 2).sum() for _, values in grouped) \
        / (len(table) - n_groups)
    return (between - within) / (between + (mean_replicates - 1) * within)


def _stride_feature_table(records: Sequence[AthleteRecord], max_strides: int
                          ) -> pd.DataFrame:
    """One row per stride of scalar features, for the nearest `max_strides`."""
    rows = []
    for record in records:
        for start, end in record.stride_windows[:max_strides]:
            cycle = np.column_stack([
                SP.resample_to_cycle(record.raw_angles[start:end + 1, angle])
                for angle in range(record.raw_angles.shape[1])])
            rows.append({"pid": record.pid, **SP.scalar_features(cycle)})
    return pd.DataFrame(rows)


def sweep_stride_reliability(records: Sequence[AthleteRecord],
                             stride_counts: Sequence[int]) -> pd.DataFrame:
    """ICC summary and stride-velocity spread as more strides are pooled.

    As the window reaches back from peak velocity into acceleration, the pooled
    strides stop being repeats of one movement and ICC correctly collapses.
    """
    rows = []
    for max_strides in stride_counts:
        table = _stride_feature_table(records, max_strides)
        feature_columns = [column for column in table.columns if column != "pid"]
        iccs = {column: icc_one_way(table, column) for column in feature_columns}

        velocity_pct, within_spread = [], []
        for record in records:
            curve = record.smoothed_velocity_curve
            peak = curve.max()
            pcts = np.array([
                curve[min(int((start + end) / 2), len(curve) - 1)] / peak * 100
                for start, end in record.stride_windows[:max_strides]])
            velocity_pct.append(pcts)
            within_spread.append(pcts.max() - pcts.min())
        pooled = np.concatenate(velocity_pct)

        rows.append({
            "strides": "all" if max_strides == 999 else max_strides,
            "n_strides": len(table),
            "icc_above_075": sum(value > 0.75 for value in iccs.values()),
            "icc_below_050": sum(value < 0.50 for value in iccs.values()),
            "median_icc": float(np.median(list(iccs.values()))),
            "mean_pct_of_peak": float(pooled.mean()),
            "within_athlete_spread": float(np.mean(within_spread)),
        })
    return pd.DataFrame(rows)


reliability = sweep_stride_reliability(cohort, STRIDE_COUNT_SWEEP)
print("[L7] reliability and stride-velocity spread as strides are pooled:")
print(reliability.to_string(index=False,
      formatters={"median_icc": "{:.3f}".format,
                  "mean_pct_of_peak": "{:.1f}".format,
                  "within_athlete_spread": "{:.1f}".format}))
print("[L7] past ~8 strides the window reaches into acceleration; the 'repeats'")
print("[L7] are no longer repeats, so pooling them destroys between-athlete signal.")
reliability.to_csv(RESULTS_DIR / "l7_stride_reliability.csv", index=False)

[L7] reliability and stride-velocity spread as strides are pooled:
strides  n_strides  icc_above_075  icc_below_050 median_icc mean_pct_of_peak within_athlete_spread
      5        150             16              1      0.802             99.4                   1.4
      8        240              7              5      0.710             99.0                   3.0
     10        300              1              5      0.640             98.4                   5.7
    all        497              0             12      0.359             91.6                  44.7
[L7] past ~8 strides the window reaches into acceleration; the 'repeats'
[L7] are no longer repeats, so pooling them destroys between-athlete signal.


### [L8] Averaging steps that start on different feet

`find_first_steps` returns contact-to-contact of **alternate** feet, and the
cycle averager applies no side swap or phase roll, so a right-limb curve starts
in stance in one window and in swing in the next. The control uses steps 1-3-5,
all starting on the **same** foot; they span a *wider* stretch of the run, so if
step-to-step progression were the cause the spread should rise, not fall.

In [17]:
def compare_step_parity_spread(records: Sequence[AthleteRecord]) -> Dict[str, float]:
    """Phase-spread of peak knee flexion for three cycle definitions.

    Returns:
        Median spread for top-speed same-foot strides, alternating accel steps
        (current pipeline), and same-parity accel steps (the fix).
    """
    top = [record.top_phase_spread for record in records]
    alternating = [record.accel_alt_spread for record in records
                   if record.accel_alt_spread is not None]
    same_parity = [record.accel_par_spread for record in records
                   if record.accel_par_spread is not None]
    return {
        "top_median": float(np.median(top)),
        "alternating_median": float(np.median(alternating)),
        "same_parity_median": float(np.median(same_parity)),
        "n_alternating": len(alternating),
        "n_same_parity": len(same_parity),
    }


parity = compare_step_parity_spread(cohort)
print("[L8] phase spread of peak knee flexion across averaged cycles (median %):")
print(f"[L8]   top speed, same-foot strides   {parity['top_median']:5.0f} %")
print(f"[L8]   accel, ALTERNATING 1-2-3       {parity['alternating_median']:5.0f} %"
      f"  (n={parity['n_alternating']})")
print(f"[L8]   accel, SAME-PARITY 1-3-5       {parity['same_parity_median']:5.0f} %"
      f"  (n={parity['n_same_parity']})")
print("[L8] the wider-spanning same-parity set is the CLEANER one, so the spread")
print("[L8] was the alternation, not the progression it has been attributed to.")

[L8] phase spread of peak knee flexion across averaged cycles (median %):
[L8]   top speed, same-foot strides       3 %
[L8]   accel, ALTERNATING 1-2-3          90 %  (n=30)
[L8]   accel, SAME-PARITY 1-3-5          14 %  (n=30)
[L8] the wider-spanning same-parity set is the CLEANER one, so the spread
[L8] was the alternation, not the progression it has been attributed to.


## Level 3 - Statistical parametric mapping

A range of motion is one number for a whole curve and cannot say *when* fast and
slow athletes diverge. `SP.spm_correlation` correlates velocity against the angle
at each of the 101 phase points, thresholds the t curve, and tests each
contiguous cluster against a null built by shuffling the velocity labels
`N_PERMUTATIONS` times. The angle set is pre-specified.

### [S1] Where stance ends

In [18]:
def locate_toe_off(records: Sequence[AthleteRecord]) -> int:
    """First cycle point (percent) at which the cohort is majority airborne."""
    stance_share = np.array([record.stance_mask for record in records]).mean(axis=0)
    return int(np.where(stance_share < 0.5)[0].min())


toe_off = locate_toe_off(cohort)
mean_stance = np.array([record.stance_mask.sum() for record in cohort]).mean()
print(f"[S1] stance = 0-{toe_off - 1}% of the cycle, swing = {toe_off}-100%")
print(f"[S1] mean stance length {mean_stance:.0f} of 101 points")

[S1] stance = 0-17% of the cycle, swing = 18-100%
[S1] mean stance length 19 of 101 points


### [S2] Top speed, pre-specified angles

In [19]:
top_speed_spm_table, top_speed_spm_detail = SP.spm_report(
    smoothed_cycle_map(cohort), target_by_pid(cohort),
    list(PRESPECIFIED_SPM_ANGLES), n_perm=N_PERMUTATIONS, seed=1)
top_speed_spm_table.to_csv(RESULTS_DIR / "s2_spm_top_speed.csv", index=False)
print(f"\n[S2] clusters sit in swing (>{toe_off}% of the cycle), consistent with a")
print("[S2] swing-fold rather than a stance-collapse interpretation of knee ROM.")

SPM, 5000 label shuffles, cluster-mass inference (alpha = 0.05, n = 30):
     angle cluster  peak_r  mass      p
    knee_R 17-29 %   0.438  31.0 0.1470
    knee_R 48-66 %  -0.541  55.0 0.0578
     hip_R 24-46 %   0.432  54.4 0.0708
trunk_lean    none     NaN   NaN    NaN

[S2] clusters sit in swing (>18% of the cycle), consistent with a
[S2] swing-fold rather than a stance-collapse interpretation of knee ROM.


### [S3] What an uncorrected pointwise test would have claimed

In [20]:
def count_uncorrected_hits(records: Sequence[AthleteRecord], angle: str,
                           alpha: float = SPM_ALPHA) -> Dict[str, float]:
    """Pointwise significant-point count and peak |r| with no cluster correction.

    Returns:
        Dict with the number of the 101 points reaching p < alpha and the
        largest absolute correlation - what an uncorrected SPM map would shade.
    """
    index = SP.ANGLE_NAMES.index(angle)
    curves = np.array([record.smoothed_cycle[:, index] for record in records])
    velocity = velocity_vector(records)
    pointwise = [stats.pearsonr(curves[:, point], velocity)
                 for point in range(curves.shape[1])]
    correlations = np.array([value[0] for value in pointwise])
    p_values = np.array([value[1] for value in pointwise])
    return {"n_significant": int((p_values < alpha).sum()),
            "max_abs_r": float(np.abs(correlations).max())}


uncorrected = count_uncorrected_hits(cohort, "knee_R")
knee_clusters = top_speed_spm_detail["knee_R"]["clusters"]
best_cluster_p = min((cluster["p"] for cluster in knee_clusters), default=float("nan"))
print(f"[S3] knee_R uncorrected: {uncorrected['n_significant']}/101 points reach "
      f"p<0.05 (|r| up to {uncorrected['max_abs_r']:.3f})")
print(f"[S3] knee_R cluster-corrected: best cluster p = {best_cluster_p:.4f}")
print("[S3] the uncorrected map would report a wide significant band; the cluster")
print("[S3] test shows it does not clear the threshold at n = 30.")

[S3] knee_R uncorrected: 32/101 points reach p<0.05 (|r| up to 0.541)
[S3] knee_R cluster-corrected: best cluster p = 0.0578
[S3] the uncorrected map would report a wide significant band; the cluster
[S3] test shows it does not clear the threshold at n = 30.


### [S4] Acceleration, before and after the [L8] fix

The same test on both acceleration codings, and the scalar model each produces.
This is what the alternation artefact was costing.

In [21]:
def smooth_accel_map(records: Sequence[AthleteRecord], use_parity: bool
                     ) -> Dict[str, np.ndarray]:
    """pid -> smoothed acceleration cycle for one coding.

    Args:
        use_parity: True for same-parity steps 1-3-5, False for alternating 1-2-3.
    """
    selected = {record.pid: (record.accel_par_cycle if use_parity
                             else record.accel_alt_cycle)
                for record in records
                if (record.accel_par_cycle if use_parity
                    else record.accel_alt_cycle) is not None}
    pids = sorted(selected)
    smoothed = SP.smooth_angle_curves(np.stack([selected[pid] for pid in pids]),
                                      penalty=SMOOTHING_PENALTY)
    return dict(zip(pids, smoothed))


def acceleration_scalar_model(records: Sequence[AthleteRecord], use_parity: bool
                              ) -> Dict[str, float]:
    """Trunk-lean + height LOO R2 for one acceleration coding."""
    selected = [record for record in records
                if (record.accel_par_cycle if use_parity
                    else record.accel_alt_cycle) is not None]
    trunk_lean = np.array([
        SP.scalar_features(record.accel_par_cycle if use_parity
                           else record.accel_alt_cycle)["trunk_lean_mean"]
        for record in selected])
    height = np.array([record.body_height_m for record in selected])
    velocity = np.array([record.peak_velocity_ms for record in selected])
    design = np.column_stack([trunk_lean, height])
    correlation, p_value = stats.pearsonr(trunk_lean, velocity)
    return {"n": len(selected), "loo_r2": SP.loo_r2_fast(design, velocity),
            "r_trunk_lean": correlation, "p_trunk_lean": p_value}


for use_parity, label in [(False, "alternating steps 1-2-3 (current pipeline)"),
                          (True, "same-parity steps 1-3-5 (corrected)")]:
    print(f"\n[S4] acceleration SPM - {label}")
    SP.spm_report(smooth_accel_map(cohort, use_parity), target_by_pid(cohort),
                  list(PRESPECIFIED_SPM_ANGLES), n_perm=N_PERMUTATIONS, seed=1)

print("\n[S4] trunk-lean + height acceleration model:")
for use_parity, label in [(False, "alternating 1-2-3"), (True, "same-parity 1-3-5")]:
    model = acceleration_scalar_model(cohort, use_parity)
    print(f"[S4]   {label:20} n={model['n']}  LOO R2 {model['loo_r2']:+.3f}  "
          f"r(trunk lean, v) {model['r_trunk_lean']:+.3f} p={model['p_trunk_lean']:.4f}")
print("[S4] correcting the artefact turns fragmented non-significant clusters into")
print("[S4] one significant swing-long trunk-lean effect and lifts the model.")


[S4] acceleration SPM - alternating steps 1-2-3 (current pipeline)


SPM, 5000 label shuffles, cluster-mass inference (alpha = 0.05, n = 30):
     angle  cluster  peak_r  mass      p
    knee_R     none     NaN   NaN    NaN
     hip_R    0-2 %   0.375   6.3 0.1344
     hip_R 93-100 %   0.403  17.5 0.1160
trunk_lean   1-11 %   0.366  22.8 0.0718
trunk_lean 69-100 %   0.480  76.6 0.0518

[S4] acceleration SPM - same-parity steps 1-3-5 (corrected)


SPM, 5000 label shuffles, cluster-mass inference (alpha = 0.05, n = 30):
     angle cluster  peak_r  mass      p
    knee_R    none     NaN   NaN    NaN
     hip_R    none     NaN   NaN    NaN
trunk_lean 0-100 %   0.534 248.0 0.0184

[S4] trunk-lean + height acceleration model:
[S4]   alternating 1-2-3    n=30  LOO R2 +0.231  r(trunk lean, v) +0.367 p=0.0461
[S4]   same-parity 1-3-5    n=30  LOO R2 +0.268  r(trunk lean, v) +0.427 p=0.0187
[S4] correcting the artefact turns fragmented non-significant clusters into
[S4] one significant swing-long trunk-lean effect and lifts the model.


## Level 4 - What survives

Every row leave-one-out, every preprocessing step refit inside the fold. The
reference is not zero: leave-one-out makes even the cohort mean slightly worse
than the full-sample mean, so an intercept-only model scores below zero.

### [W1] The honest model ladder

In [22]:
def build_honest_ladder(features: pd.DataFrame, target: np.ndarray,
                        height: np.ndarray) -> Tuple[pd.DataFrame, np.ndarray]:
    """Leave-one-out R2 for the model ladder, plus the winning design matrix.

    Returns:
        (ladder table, knee-ROM+height design matrix) so the permutation test in
        the next cell can reuse the winning model.
    """
    n = len(target)
    knee_rom = features[["knee_hi_ROM", "knee_lo_ROM"]].values
    knee_rom_height = np.column_stack([knee_rom, height])
    specifications = [
        ("intercept only", np.zeros((n, 1)), None),
        ("height only", height[:, None], None),
        ("knee ROM (hi + lo)", knee_rom, None),
        ("knee ROM + height", knee_rom_height, None),
        ("all 22 scalars, ridge", features.values, make_ridge),
    ]
    rows = []
    for label, design, model_factory in specifications:
        metrics = SP.fit_metrics(target, SP.loo_predict(design, target, model_factory))
        rows.append({"model": label, "loo_r2": metrics["LOO_R2"],
                     "rmse": metrics["RMSE"], "mae": metrics["MAE"]})
    return pd.DataFrame(rows), knee_rom_height


ladder, winning_design = build_honest_ladder(FEATURES, VELOCITY, HEIGHT)
print("[W1] honest model ladder (leave-one-out):")
print(ladder.to_string(index=False,
      formatters={"loo_r2": "{:+.3f}".format, "rmse": "{:.3f}".format,
                  "mae": "{:.3f}".format}))

observed_r2, permutation_p, permutation_null = SP.permutation_test(
    winning_design, VELOCITY, n_perm=N_PERMUTATIONS, seed=1)
print(f"\n[W1] knee ROM + height: LOO R2 {observed_r2:+.3f}, "
      f"permutation p = {permutation_p:.4f}, "
      f"null 95th pct {np.percentile(permutation_null, 95):+.3f}")
ladder.to_csv(RESULTS_DIR / "w1_honest_ladder.csv", index=False)

[W1] honest model ladder (leave-one-out):
                model loo_r2  rmse   mae
       intercept only -0.070 0.735 0.619
          height only +0.097 0.675 0.539
   knee ROM (hi + lo) +0.220 0.628 0.514
    knee ROM + height +0.256 0.613 0.471
all 22 scalars, ridge -0.227 0.788 0.638



[W1] knee ROM + height: LOO R2 +0.256, permutation p = 0.0024, null 95th pct +0.019


### [W2] The ledger, assembled

In [23]:
def assemble_ledger(ledger_rows: List[Dict[str, float]],
                    surviving_label: str, surviving_r2: float) -> pd.DataFrame:
    """Combine every shortcut's reported/honest pair with the surviving model."""
    table = pd.DataFrame(ledger_rows)
    table = pd.concat([table, pd.DataFrame([{
        "shortcut": surviving_label, "reported": surviving_r2,
        "honest": surviving_r2, "inflation": 0.0}])], ignore_index=True)
    return table


ledger = assemble_ledger(LEDGER_ROWS, "pre-specified knee ROM + height", observed_r2)
print("[W2] the leakage ledger:")
print(ledger.to_string(index=False,
      formatters={c: "{:+.3f}".format for c in
                  ["reported", "honest", "inflation"]}))
print("\n[W2] the pre-specified three-column model beats the exhaustively searched")
print("[W2] one once both are scored honestly. That is the whole argument.")
ledger.to_csv(RESULTS_DIR / "w2_leakage_ledger.csv", index=False)

[W2] the leakage ledger:
                            shortcut reported honest inflation
top-3 univariate selection on all 30   +0.228 +0.072    +0.156
      exhaustive search, 1540 models   +0.403 -0.020    +0.423
          fPCA basis on all 30 (R/L)   -0.223 -0.188    -0.036
     pre-specified knee ROM + height   +0.256 +0.256    +0.000

[W2] the pre-specified three-column model beats the exhaustively searched
[W2] one once both are scored honestly. That is the whole argument.


## Summary and verification

A final recomputation of the numbers the write-up depends on, so the notebook
fails loudly if an upstream change moves any of them.

In [24]:
print("=" * 66)
print("VALIDATION SUMMARY")
print("=" * 66)
print(f"cohort                     : {len(cohort)} athletes, "
      f"{VELOCITY.min():.2f}-{VELOCITY.max():.2f} m/s")
print(f"[V1] knee peak missed 60Hz : {sampling_loss.missed_deg.mean():.2f} deg, "
      f"r(miss, speed) = {r_velocity:+.3f}  -> definition offset, not artefact")
print(f"[V2] knee ROM vs velocity  : 2-D {r_2d:+.3f} | 3-D {r_3d:+.3f}  -> robust")
print(f"[L1] top-3 selection leak  : "
      f"{selection_leakage.set_index('k').loc[3, 'inflation']:+.3f}")
print(f"[L2] model-search leak     : "
      f"{model_search['reported'] - model_search['honest']:+.3f}")
print(f"[L4] PC1 {pc1['variance_share']:.0f}% variance   : "
      f"r(velocity) = {pc1['velocity_correlation']:+.3f}  -> variance != prediction")
print(f"[L6] window-length artefact: {int((window_artefact.p_raw < 0.05).sum())}/7 "
      f"significant -> {int(window_artefact.survives.sum())}/7 survive")
print(f"[L8] accel phase spread    : alt {parity['alternating_median']:.0f}% -> "
      f"parity {parity['same_parity_median']:.0f}%")
print(f"[W1] honest headline       : LOO R2 {observed_r2:+.3f}, p = {permutation_p:.4f}")
print("=" * 66)

_expected = {"cohort size": len(cohort) == 30,
             "knee miss < 1 deg": sampling_loss.missed_deg.mean() < 1.0,
             "headline in 0.20-0.31": 0.20 < observed_r2 < 0.31,
             "parity spread < alt spread":
                 parity["same_parity_median"] < parity["alternating_median"]}
for description, passed in _expected.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {description}")
assert all(_expected.values()), "a verification check failed"
print("\nAll verification checks passed.")

VALIDATION SUMMARY
cohort                     : 30 athletes, 6.39-9.35 m/s
[V1] knee peak missed 60Hz : 0.21 deg, r(miss, speed) = -0.000  -> definition offset, not artefact
[V2] knee ROM vs velocity  : 2-D -0.565 | 3-D -0.617  -> robust
[L1] top-3 selection leak  : +0.156
[L2] model-search leak     : +0.423
[L4] PC1 42% variance   : r(velocity) = -0.177  -> variance != prediction
[L6] window-length artefact: 6/7 significant -> 0/7 survive
[L8] accel phase spread    : alt 90% -> parity 14%
[W1] honest headline       : LOO R2 +0.256, p = 0.0024
  [PASS] cohort size
  [PASS] knee miss < 1 deg
  [PASS] headline in 0.20-0.31
  [PASS] parity spread < alt spread

All verification checks passed.
